# 0

In [10]:
import pandas as pd
import numpy as np
drought = pd.read_csv('/content/drought_dataset.csv')
desease = pd.read_csv('/content/disease_dataset.csv')
rain = pd.read_csv('/content/rain_dataset.csv')

# 1. Setting

## 1. Configuration initiale et création des dossiers

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, accuracy_score, f1_score, classification_report, confusion_matrix, roc_curve, auc
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from xgboost import XGBRegressor, XGBClassifier
from lightgbm import LGBMRegressor, LGBMClassifier
import joblib
import os
from datetime import datetime

# Configuration des styles pour les graphiques
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Création des dossiers de résultats
def create_folders():
    """Crée l'arborescence des dossiers pour sauvegarder les résultats"""
    base_dir = "/content/result"
    folders = [
        base_dir,
        f"{base_dir}/models",
        f"{base_dir}/rain",
        f"{base_dir}/disease",
        f"{base_dir}/drought"
    ]

    for folder in folders:
        os.makedirs(folder, exist_ok=True)
        print(f"Dossier créé: {folder}")

    return base_dir

# Création des dossiers
BASE_DIR = create_folders()

Dossier créé: /content/result
Dossier créé: /content/result/models
Dossier créé: /content/result/rain
Dossier créé: /content/result/disease
Dossier créé: /content/result/drought


## 2. Fonctions de visualisation

In [13]:
def plot_regression_results(y_true, y_pred, model_name, save_path):
    """Graphiques pour les modèles de régression"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle(f'Résultats du modèle - {model_name}', fontsize=16)

    # Graphique 1: Prédictions vs Réalité
    axes[0, 0].scatter(y_true, y_pred, alpha=0.5)
    axes[0, 0].plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', lw=2)
    axes[0, 0].set_xlabel('Valeurs Réelles')
    axes[0, 0].set_ylabel('Prédictions')
    axes[0, 0].set_title('Prédictions vs Réalité')
    axes[0, 0].grid(True)

    # Graphique 2: Distribution des résidus
    residuals = y_true - y_pred
    axes[0, 1].hist(residuals, bins=30, alpha=0.7, edgecolor='black')
    axes[0, 1].axvline(0, color='red', linestyle='--')
    axes[0, 1].set_xlabel('Résidus')
    axes[0, 1].set_ylabel('Fréquence')
    axes[0, 1].set_title('Distribution des Résidus')
    axes[0, 1].grid(True)

    # Graphique 3: Résidus vs Prédictions
    axes[1, 0].scatter(y_pred, residuals, alpha=0.5)
    axes[1, 0].axhline(0, color='red', linestyle='--')
    axes[1, 0].set_xlabel('Prédictions')
    axes[1, 0].set_ylabel('Résidus')
    axes[1, 0].set_title('Résidus vs Prédictions')
    axes[1, 0].grid(True)

    # Graphique 4: Évolution des prédictions
    axes[1, 1].plot(y_true.values[:100], label='Réel', alpha=0.7)
    axes[1, 1].plot(y_pred[:100], label='Prédit', alpha=0.7)
    axes[1, 1].set_xlabel('Échantillons')
    axes[1, 1].set_ylabel('Valeur')
    axes[1, 1].set_title('Comparaison Réel vs Prédit (100 premiers)')
    axes[1, 1].legend()
    axes[1, 1].grid(True)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

def plot_classification_results(y_true, y_pred, y_proba, model_name, save_path):
    """Graphiques pour les modèles de classification"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle(f'Résultats du modèle - {model_name}', fontsize=16)

    # Matrice de confusion
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0])
    axes[0, 0].set_xlabel('Prédit')
    axes[0, 0].set_ylabel('Réel')
    axes[0, 0].set_title('Matrice de Confusion')

    # Courbe ROC
    if y_proba is not None:
        fpr, tpr, _ = roc_curve(y_true, y_proba)
        roc_auc = auc(fpr, tpr)
        axes[0, 1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
        axes[0, 1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
        axes[0, 1].set_xlim([0.0, 1.0])
        axes[0, 1].set_ylim([0.0, 1.05])
        axes[0, 1].set_xlabel('Taux Faux Positifs')
        axes[0, 1].set_ylabel('Taux Vrais Positifs')
        axes[0, 1].set_title('Courbe ROC')
        axes[0, 1].legend(loc="lower right")
        axes[0, 1].grid(True)

    # Distribution des probabilités
    if y_proba is not None:
        axes[1, 0].hist(y_proba, bins=30, alpha=0.7, edgecolor='black')
        axes[1, 0].set_xlabel('Probabilité Prédite')
        axes[1, 0].set_ylabel('Fréquence')
        axes[1, 0].set_title('Distribution des Probabilités')
        axes[1, 0].grid(True)

    # Importance des features (sera rempli plus tard)
    axes[1, 1].text(0.5, 0.5, 'Importance des features\n(à ajouter après entraînement)',
                    ha='center', va='center', transform=axes[1, 1].transAxes, fontsize=12)
    axes[1, 1].set_title('Importance des Features')

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

def plot_learning_curve(model, X, y, model_name, save_path):
    """Courbe d'apprentissage"""
    train_sizes, train_scores, test_scores = learning_curve(
        model, X, y, cv=5, n_jobs=-1,
        train_sizes=np.linspace(0.1, 1.0, 10),
        scoring='neg_mean_squared_error' if hasattr(model, 'predict') else 'accuracy'
    )

    train_scores_mean = -np.mean(train_scores, axis=1) if hasattr(model, 'predict') else np.mean(train_scores, axis=1)
    test_scores_mean = -np.mean(test_scores, axis=1) if hasattr(model, 'predict') else np.mean(test_scores, axis=1)

    plt.figure(figsize=(10, 6))
    plt.plot(train_sizes, train_scores_mean, 'o-', color='r', label='Score entraînement')
    plt.plot(train_sizes, test_scores_mean, 'o-', color='g', label='Score validation')
    plt.xlabel("Taille de l'ensemble d'entraînement")
    plt.ylabel('Score (MSE)' if hasattr(model, 'predict') else 'Score (Accuracy)')
    plt.title(f'Courbe d\'apprentissage - {model_name}')
    plt.legend(loc='best')
    plt.grid(True)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

def plot_feature_importance(model, feature_names, model_name, save_path):
    """Importance des features"""
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        indices = np.argsort(importances)[::-1]

        plt.figure(figsize=(10, 8))
        plt.title(f'Importance des Features - {model_name}')
        plt.bar(range(min(15, len(importances))), importances[indices[:15]])
        plt.xticks(range(min(15, len(importances))), [feature_names[i] for i in indices[:15]], rotation=45)
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()

## 3. Fonctions de prétraitement et d'évaluation

In [14]:
def preprocess_data(df, target_column):
    """Fonction de prétraitement commune"""
    # Supprimer les valeurs manquantes
    df = df.dropna()

    # Encodage cyclique des variables temporelles
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['day_sin'] = np.sin(2 * np.pi * df['day'] / 31)
    df['day_cos'] = np.cos(2 * np.pi * df['day'] / 31)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

    # Supprimer les colonnes temporelles originales
    df = df.drop(['observation_time', 'hour', 'day', 'month'], axis=1, errors='ignore')

    return df

def evaluate_regression_models(X_train, X_test, y_train, y_test, model_name):
    """Évaluation des modèles de régression"""
    models = {
        'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42),
        'XGBoost': XGBRegressor(random_state=42),
        'LightGBM': LGBMRegressor(random_state=42),
        'LinearRegression': LinearRegression()
    }

    results = {}
    for name, model in models.items():
        print(f"Entraînement de {name}...")
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        results[name] = {'model': model, 'rmse': rmse, 'mse': mse, 'predictions': y_pred}

        # Graphiques pour chaque modèle
        plot_regression_results(y_test, y_pred, f"{model_name} - {name}",
                              f"{BASE_DIR}/{model_name.lower()}/{name}_results.png")

        # Courbe d'apprentissage
        plot_learning_curve(model, X_train, y_train, f"{model_name} - {name}",
                          f"{BASE_DIR}/{model_name.lower()}/{name}_learning_curve.png")

        # Importance des features
        plot_feature_importance(model, [f"Feature_{i}" for i in range(X_train.shape[1])],
                              f"{model_name} - {name}",
                              f"{BASE_DIR}/{model_name.lower()}/{name}_feature_importance.png")

    return results

def evaluate_classification_models(X_train, X_test, y_train, y_test, model_name):
    """Évaluation des modèles de classification"""
    models = {
        'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
        'XGBoost': XGBClassifier(random_state=42),
        'LightGBM': LGBMClassifier(random_state=42),
        'LogisticRegression': LogisticRegression(random_state=42)
    }

    results = {}
    for name, model in models.items():
        print(f"Entraînement de {name}...")
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Probabilités pour la courbe ROC
        y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None

        accuracy = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')
        results[name] = {'model': model, 'accuracy': accuracy, 'f1': f1,
                        'predictions': y_pred, 'probabilities': y_proba}

        # Graphiques pour chaque modèle
        plot_classification_results(y_test, y_pred, y_proba, f"{model_name} - {name}",
                                  f"{BASE_DIR}/{model_name.lower()}/{name}_results.png")

        # Courbe d'apprentissage
        plot_learning_curve(model, X_train, y_train, f"{model_name} - {name}",
                          f"{BASE_DIR}/{model_name.lower()}/{name}_learning_curve.png")

        # Importance des features
        if hasattr(model, 'feature_importances_'):
            plot_feature_importance(model, [f"Feature_{i}" for i in range(X_train.shape[1])],
                                  f"{model_name} - {name}",
                                  f"{BASE_DIR}/{model_name.lower()}/{name}_feature_importance.png")

    return results

## 4. Configuration initiale MLflow

In [3]:
!pip install dagshub mlflow --quiet


In [24]:
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature
import dagshub
import os

# Initialisation de DagsHub avec MLflow
print("🔗 Initialisation de DagsHub...")
os.environ["MLFLOW_TRACKING_USERNAME"] = "IbrahimFaye"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "e87c03585d0c21bc7082c657eb58c2361d196d75"
dagshub.init(repo_owner='IbrahimFaye', repo_name='weather-agri', mlflow=True)

# Configuration MLflow
mlflow.set_tracking_uri('https://dagshub.com/IbrahimFaye/weather-agri.mlflow')

🔗 Initialisation de DagsHub...


Initialized MLflow to track repo "IbrahimFaye/weather-agri"

Repository IbrahimFaye/weather-agri initialized!

# 2. MODELS

## 2. Fonctions utilitaires MLflow

In [39]:
def log_classification_metrics(y_true, y_pred, y_proba=None):
    """Log toutes les métriques de classification dans MLflow"""
    from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)

    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)

    if y_proba is not None:
        try:
            auc = roc_auc_score(y_true, y_proba)
            mlflow.log_metric("auc", auc)
        except:
            pass

    return {"accuracy": accuracy, "f1": f1, "precision": precision, "recall": recall}

def setup_mlflow_experiment(experiment_name):
    """Configure l'expérience MLflow"""
    mlflow.set_experiment(experiment_name)
    print(f"📊 Expérience MLflow configurée: {experiment_name}")

## 3. Modèle de pluie avec MLflow

In [35]:
def train_rain_model(rain_df):
    """Entraînement du modèle de prédiction de pluie avec MLflow et gestion des poids"""
    print("=== PRÉDICTION DE PLUIE (CLASSIFICATION BINAIRE) ===")

    # Configuration MLflow
    setup_mlflow_experiment("rain_forecast")

    # Prétraitement
    df = preprocess_data(rain_df, 'rain_label')

    # Séparation features/target
    X = df.drop(['rain_next_hour', 'rain_label'], axis=1)
    y = df['rain_label']
    feature_names = X.columns.tolist()

    # Analyse du déséquilibre des classes
    class_counts = y.value_counts()
    total_samples = len(y)
    class_0_ratio = class_counts[0] / total_samples
    class_1_ratio = class_counts[1] / total_samples

    print(f"📊 Distribution des classes: {class_counts.to_dict()}")
    print(f"📈 Ratio des classes: Classe 0 (pas de pluie): {class_0_ratio:.2%}, Classe 1 (pluie): {class_1_ratio:.2%}")

    # Calcul des poids de classe
    class_weights = {
        0: total_samples / (2 * class_counts[0]),
        1: total_samples / (2 * class_counts[1])
    }

    # Pour XGBoost: calcul de scale_pos_weight
    scale_pos_weight = class_counts[0] / class_counts[1]

    print(f"⚖️ Poids de classe calculés: {class_weights}")
    print(f"📏 scale_pos_weight pour XGBoost: {scale_pos_weight:.2f}")

    # Split train/test avec stratification
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    # Normalisation
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Modèles à évaluer avec gestion des poids
    models = {
        'RandomForest': RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            class_weight=class_weights
        ),
        'XGBoost': XGBClassifier(
            random_state=42,
            scale_pos_weight=scale_pos_weight,
            eval_metric='logloss'
        ),
        'LightGBM': LGBMClassifier(
            random_state=42,
            class_weight=class_weights,
            verbose=-1
        ),
        'LogisticRegression': LogisticRegression(
            random_state=42,
            class_weight=class_weights,
            max_iter=1000
        )
    }

    results = {}

    for name, model in models.items():
        print(f"\n🔧 Entraînement de {name} avec gestion des poids...")

        # Début du run MLflow
        with mlflow.start_run(run_name=name, nested=True):
            # Log des informations sur le déséquilibre
            mlflow.log_param("class_0_count", class_counts[0])
            mlflow.log_param("class_1_count", class_counts[1])
            mlflow.log_param("class_0_ratio", class_0_ratio)
            mlflow.log_param("class_1_ratio", class_1_ratio)

            # Log des poids spécifiques selon le modèle
            if name == 'XGBoost':
                mlflow.log_param("scale_pos_weight", scale_pos_weight)
            else:
                mlflow.log_param("class_weight_0", class_weights[0])
                mlflow.log_param("class_weight_1", class_weights[1])

            # Entraînement du modèle
            model.fit(X_train_scaled, y_train)
            y_pred = model.predict(X_test_scaled)
            y_proba = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, 'predict_proba') else None

            # Log des paramètres du modèle
            if hasattr(model, 'get_params'):
                mlflow.log_params(model.get_params())

            # Log des métriques détaillées
            from sklearn.metrics import precision_score, recall_score, roc_auc_score, f1_score, accuracy_score

            accuracy = accuracy_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred)
            precision = precision_score(y_test, y_pred)
            recall = recall_score(y_test, y_pred)

            # Métriques spécifiques pour la classe minoritaire (pluie)
            f1_class_1 = f1_score(y_test, y_pred, pos_label=1)
            recall_class_1 = recall_score(y_test, y_pred, pos_label=1)
            precision_class_1 = precision_score(y_test, y_pred, pos_label=1)

            mlflow.log_metric("accuracy", accuracy)
            mlflow.log_metric("f1_score", f1)
            mlflow.log_metric("f1_score_class_1", f1_class_1)
            mlflow.log_metric("precision", precision)
            mlflow.log_metric("precision_class_1", precision_class_1)
            mlflow.log_metric("recall", recall)
            mlflow.log_metric("recall_class_1", recall_class_1)

            if y_proba is not None:
                try:
                    auc = roc_auc_score(y_test, y_proba)
                    mlflow.log_metric("auc", auc)
                except:
                    pass

            results[name] = {
                'model': model,
                'metrics': {
                    'accuracy': accuracy,
                    'f1': f1,
                    'f1_class_1': f1_class_1,
                    'precision': precision,
                    'precision_class_1': precision_class_1,
                    'recall': recall,
                    'recall_class_1': recall_class_1
                },
                'predictions': y_pred,
                'probabilities': y_proba
            }

            print(f"✅ {name} - Accuracy: {accuracy:.4f}, F1: {f1:.4f}, F1_Pluie: {f1_class_1:.4f}")
            print(f"   Rappel Pluie: {recall_class_1:.4f}, Précision Pluie: {precision_class_1:.4f}")

    # Sélection du meilleur modèle basé sur F1 de la classe 1 (pluie)
    best_model_name = max(results.keys(), key=lambda x: results[x]['metrics']['f1_class_1'])
    best_model = results[best_model_name]['model']
    best_metrics = results[best_model_name]['metrics']

    # Run pour le meilleur modèle
    with mlflow.start_run(run_name="BEST_MODEL") as run:
        # Log des informations sur le déséquilibre
        mlflow.log_param("class_0_count", class_counts[0])
        mlflow.log_param("class_1_count", class_counts[1])
        mlflow.log_param("class_0_ratio", class_0_ratio)
        mlflow.log_param("class_1_ratio", class_1_ratio)

        # Log des poids utilisés
        if best_model_name == 'XGBoost':
            mlflow.log_param("scale_pos_weight", scale_pos_weight)
        else:
            mlflow.log_param("class_weight_0", class_weights[0])
            mlflow.log_param("class_weight_1", class_weights[1])

        mlflow.log_params(best_model.get_params())

        # Log de toutes les métriques
        for metric_name, metric_value in best_metrics.items():
            mlflow.log_metric(metric_name, metric_value)

        # Log des informations sur les données
        mlflow.log_param("dataset_size", len(df))
        mlflow.log_param("n_features", X.shape[1])
        mlflow.log_param("test_size", len(X_test))
        mlflow.log_param("best_model", best_model_name)

        print(f"🎯 Meilleur modèle MLflow: {best_model_name}")
        print(f"📊 Performance sur la pluie - F1: {best_metrics['f1_class_1']:.4f}, Rappel: {best_metrics['recall_class_1']:.4f}")
        print(f"🔗 Run ID: {run.info.run_id}")

    # Sauvegarde locale
    joblib.dump(best_model, f'{BASE_DIR}/models/rain_model.pkl')
    joblib.dump(scaler, f'{BASE_DIR}/models/rain_scaler.pkl')
    joblib.dump(feature_names, f'{BASE_DIR}/models/rain_features.pkl')
    joblib.dump(class_weights, f'{BASE_DIR}/models/rain_class_weights.pkl')

    return best_model, scaler, feature_names, class_weights

## 4. Modèle de maladie avec MLflow

In [37]:
def train_disease_model(disease_df):
    """Entraînement du modèle de prédiction de maladie avec MLflow"""
    print("\n=== PRÉDICTION DE MALADIE (CLASSIFICATION) ===")

    # Configuration MLflow
    setup_mlflow_experiment("disease_risk_prediction")

    # Prétraitement
    df = preprocess_data(disease_df, 'disease_risk')

    # Séparation features/target
    X = df.drop('disease_risk', axis=1)
    y = df['disease_risk']
    feature_names = X.columns.tolist()

    print(f"Distribution des classes: {y.value_counts()}")

    # Split train/test avec stratification
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    # Normalisation
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Modèles à évaluer
    models = {
        'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
        'XGBoost': XGBClassifier(random_state=42),
        'LightGBM': LGBMClassifier(random_state=42),
        'LogisticRegression': LogisticRegression(random_state=42)
    }

    results = {}

    for name, model in models.items():
        print(f"\n🔧 Entraînement de {name}...")

        # Début du run MLflow
        with mlflow.start_run(run_name=name, nested=True):
            # Entraînement du modèle
            model.fit(X_train_scaled, y_train)
            y_pred = model.predict(X_test_scaled)
            y_proba = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, 'predict_proba') else None

            # Log des paramètres
            if hasattr(model, 'get_params'):
                mlflow.log_params(model.get_params())

            # Log des métriques
            metrics = log_classification_metrics(y_test, y_pred, y_proba)

            results[name] = {
                'model': model,
                'metrics': metrics,
                'predictions': y_pred,
                'probabilities': y_proba
            }

            print(f"✅ {name} - Accuracy: {metrics['accuracy']:.4f}, F1: {metrics['f1']:.4f}")

    # Sélection du meilleur modèle
    best_model_name = max(results.keys(), key=lambda x: results[x]['metrics']['f1'])
    best_model = results[best_model_name]['model']
    best_metrics = results[best_model_name]['metrics']

    # Run pour le meilleur modèle
    with mlflow.start_run(run_name="BEST_MODEL") as run:
        mlflow.log_params(best_model.get_params())
        for metric_name, metric_value in best_metrics.items():
            mlflow.log_metric(metric_name, metric_value)

        # Log des informations sur les données
        mlflow.log_param("dataset_size", len(df))
        mlflow.log_param("n_features", X.shape[1])
        mlflow.log_param("test_size", len(X_test))
        mlflow.log_param("class_distribution", str(y.value_counts().to_dict()))
        mlflow.log_param("best_model", best_model_name)

        print(f"🎯 Meilleur modèle MLflow: {best_model_name} avec F1 = {best_metrics['f1']:.4f}")
        print(f"🔗 Run ID: {run.info.run_id}")

    # Sauvegarde locale
    joblib.dump(best_model, f'{BASE_DIR}/models/disease_model.pkl')
    joblib.dump(scaler, f'{BASE_DIR}/models/disease_scaler.pkl')
    joblib.dump(feature_names, f'{BASE_DIR}/models/disease_features.pkl')

    return best_model, scaler, feature_names

## 5. Modèle de sécheresse avec MLflow

In [38]:
def train_drought_model(drought_df):
    """Entraînement du modèle de prédiction de sécheresse avec MLflow"""
    print("\n=== PRÉDICTION DE SÉCHERESSE (CLASSIFICATION) ===")

    # Configuration MLflow
    setup_mlflow_experiment("drought_forecast")

    # Prétraitement
    df = preprocess_data(drought_df, 'drought_label')

    # Séparation features/target
    X = df.drop('drought_label', axis=1)
    y = df['drought_label']
    feature_names = X.columns.tolist()

    print(f"Distribution des classes: {y.value_counts()}")

    # Split train/test avec stratification
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    # Normalisation
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Modèles à évaluer
    models = {
        'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
        'XGBoost': XGBClassifier(random_state=42),
        'LightGBM': LGBMClassifier(random_state=42),
        'LogisticRegression': LogisticRegression(random_state=42)
    }

    results = {}

    for name, model in models.items():
        print(f"\n🔧 Entraînement de {name}...")

        # Début du run MLflow
        with mlflow.start_run(run_name=name, nested=True):
            # Entraînement du modèle
            model.fit(X_train_scaled, y_train)
            y_pred = model.predict(X_test_scaled)
            y_proba = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, 'predict_proba') else None

            # Log des paramètres
            if hasattr(model, 'get_params'):
                mlflow.log_params(model.get_params())

            # Log des métriques
            metrics = log_classification_metrics(y_test, y_pred, y_proba)

            results[name] = {
                'model': model,
                'metrics': metrics,
                'predictions': y_pred,
                'probabilities': y_proba
            }

            print(f"✅ {name} - Accuracy: {metrics['accuracy']:.4f}, F1: {metrics['f1']:.4f}")

    # Sélection du meilleur modèle
    best_model_name = max(results.keys(), key=lambda x: results[x]['metrics']['f1'])
    best_model = results[best_model_name]['model']
    best_metrics = results[best_model_name]['metrics']

    # Run pour le meilleur modèle
    with mlflow.start_run(run_name="BEST_MODEL") as run:
        mlflow.log_params(best_model.get_params())
        for metric_name, metric_value in best_metrics.items():
            mlflow.log_metric(metric_name, metric_value)

        # Log des informations sur les données
        mlflow.log_param("dataset_size", len(df))
        mlflow.log_param("n_features", X.shape[1])
        mlflow.log_param("test_size", len(X_test))
        mlflow.log_param("class_distribution", str(y.value_counts().to_dict()))
        mlflow.log_param("best_model", best_model_name)

        print(f"🎯 Meilleur modèle MLflow: {best_model_name} avec F1 = {best_metrics['f1']:.4f}")
        print(f"🔗 Run ID: {run.info.run_id}")

    # Sauvegarde locale
    joblib.dump(best_model, f'{BASE_DIR}/models/drought_model.pkl')
    joblib.dump(scaler, f'{BASE_DIR}/models/drought_scaler.pkl')
    joblib.dump(feature_names, f'{BASE_DIR}/models/drought_features.pkl')

    return best_model, scaler, feature_names

## 6. Mise à jour du résumé final

In [40]:
def create_summary_report():
    """Crée un rapport récapitulatif avec informations MLflow"""
    print("\n" + "="*60)
    print("📊 RAPPORT RÉCAPITULATIF DES MODÈLES AVEC MLFLOW")
    print("="*60)

    print("\n🎯 EXPÉRIENCES MLFLOW:")
    experiments = {
        "rain_forecast": "Prédiction de pluie",
        "disease_risk_prediction": "Risque de maladie",
        "drought_forecast": "Prédiction de sécheresse"
    }

    for exp_name, exp_desc in experiments.items():
        print(f"  - {exp_name}: {exp_desc}")

    # Vérifier les fichiers sauvegardés
    model_files = os.listdir(f"{BASE_DIR}/models")
    print(f"\n💾 Modèles sauvegardés localement:")
    for file in sorted(model_files):
        print(f"  - {file}")

    print(f"\n🔗 Accès MLflow: https://dagshub.com/IbrahimFaye/weather-agri.mlflow")
    print(f"📁 Dossier local: {BASE_DIR}")
    print("="*60)

# Installation des dépendances nécessaires (à exécuter une fois)
print("📦 Installation des dépendances MLflow...")
!pip install mlflow dagshub

# Exécution de tous les modèles
print("🚀 Démarrage de l'entraînement avec MLflow...")

# Entraînement des modèles
rain_model, rain_scaler, rain_features, _ = train_rain_model(rain)
disease_model, disease_scaler, disease_features = train_disease_model(desease)
drought_model, drought_scaler, drought_features = train_drought_model(drought)

# Rapport final
create_summary_report()

📦 Installation des dépendances MLflow...
🚀 Démarrage de l'entraînement avec MLflow...
=== PRÉDICTION DE PLUIE (CLASSIFICATION BINAIRE) ===
📊 Expérience MLflow configurée: rain_forecast
📊 Distribution des classes: {0: 76437, 1: 2472}
📈 Ratio des classes: Classe 0 (pas de pluie): 96.87%, Classe 1 (pluie): 3.13%
⚖️ Poids de classe calculés: {0: np.float64(0.5161701793633973), 1: np.float64(15.960558252427184)}
📏 scale_pos_weight pour XGBoost: 30.92


/tmp/ipython-input-1760959349.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
/tmp/ipython-input-1760959349.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
/tmp/ipython-input-1760959349.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pyda


🔧 Entraînement de RandomForest avec gestion des poids...
✅ RandomForest - Accuracy: 0.9755, F1: 0.4967, F1_Pluie: 0.4967
   Rappel Pluie: 0.3866, Précision Pluie: 0.6945
🏃 View run RandomForest at: https://dagshub.com/IbrahimFaye/weather-agri.mlflow/#/experiments/1/runs/bbfcbe7b74654201a3aef3f8e082dd0a
🧪 View experiment at: https://dagshub.com/IbrahimFaye/weather-agri.mlflow/#/experiments/1

🔧 Entraînement de XGBoost avec gestion des poids...
✅ XGBoost - Accuracy: 0.9571, F1: 0.5188, F1_Pluie: 0.5188
   Rappel Pluie: 0.7389, Précision Pluie: 0.3998
🏃 View run XGBoost at: https://dagshub.com/IbrahimFaye/weather-agri.mlflow/#/experiments/1/runs/3ba9ca5b4b284f56a9762264d72a8c95
🧪 View experiment at: https://dagshub.com/IbrahimFaye/weather-agri.mlflow/#/experiments/1

🔧 Entraînement de LightGBM avec gestion des poids...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


✅ LightGBM - Accuracy: 0.9363, F1: 0.4521, F1_Pluie: 0.4521
   Rappel Pluie: 0.8401, Précision Pluie: 0.3092
🏃 View run LightGBM at: https://dagshub.com/IbrahimFaye/weather-agri.mlflow/#/experiments/1/runs/275f5075bd8d4df39472e1934b5e2960
🧪 View experiment at: https://dagshub.com/IbrahimFaye/weather-agri.mlflow/#/experiments/1

🔧 Entraînement de LogisticRegression avec gestion des poids...
✅ LogisticRegression - Accuracy: 0.8795, F1: 0.3207, F1_Pluie: 0.3207
   Rappel Pluie: 0.9089, Précision Pluie: 0.1947
🏃 View run LogisticRegression at: https://dagshub.com/IbrahimFaye/weather-agri.mlflow/#/experiments/1/runs/f016f8323ded430b9fc09e0d2eff2df1
🧪 View experiment at: https://dagshub.com/IbrahimFaye/weather-agri.mlflow/#/experiments/1
🎯 Meilleur modèle MLflow: XGBoost
📊 Performance sur la pluie - F1: 0.5188, Rappel: 0.7389
🔗 Run ID: 2bb7e66c5eef4704ad58bf1354769c4e
🏃 View run BEST_MODEL at: https://dagshub.com/IbrahimFaye/weather-agri.mlflow/#/experiments/1/runs/2bb7e66c5eef4704ad58bf1354

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


✅ LightGBM - Accuracy: 0.9990, F1: 0.9940
🏃 View run LightGBM at: https://dagshub.com/IbrahimFaye/weather-agri.mlflow/#/experiments/4/runs/5248e3c806d04620b53b128a28620cbc
🧪 View experiment at: https://dagshub.com/IbrahimFaye/weather-agri.mlflow/#/experiments/4

🔧 Entraînement de LogisticRegression...
✅ LogisticRegression - Accuracy: 0.9406, F1: 0.6051
🏃 View run LogisticRegression at: https://dagshub.com/IbrahimFaye/weather-agri.mlflow/#/experiments/4/runs/00ad7fb3796347129e995b67a633c527
🧪 View experiment at: https://dagshub.com/IbrahimFaye/weather-agri.mlflow/#/experiments/4
🎯 Meilleur modèle MLflow: RandomForest avec F1 = 1.0000
🔗 Run ID: b4b650791e454f8db5647ffb1148d664
🏃 View run BEST_MODEL at: https://dagshub.com/IbrahimFaye/weather-agri.mlflow/#/experiments/4/runs/b4b650791e454f8db5647ffb1148d664
🧪 View experiment at: https://dagshub.com/IbrahimFaye/weather-agri.mlflow/#/experiments/4

=== PRÉDICTION DE SÉCHERESSE (CLASSIFICATION) ===
📊 Expérience MLflow configurée: drought_for

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


✅ LightGBM - Accuracy: 1.0000, F1: 1.0000
🏃 View run LightGBM at: https://dagshub.com/IbrahimFaye/weather-agri.mlflow/#/experiments/3/runs/d08a909589d14f399b8572548cb90012
🧪 View experiment at: https://dagshub.com/IbrahimFaye/weather-agri.mlflow/#/experiments/3

🔧 Entraînement de LogisticRegression...
✅ LogisticRegression - Accuracy: 0.9952, F1: 0.9968
🏃 View run LogisticRegression at: https://dagshub.com/IbrahimFaye/weather-agri.mlflow/#/experiments/3/runs/26bc02a415c84b07b48e540fd1c016db
🧪 View experiment at: https://dagshub.com/IbrahimFaye/weather-agri.mlflow/#/experiments/3
🎯 Meilleur modèle MLflow: RandomForest avec F1 = 1.0000
🔗 Run ID: 4e3936edd4bc40e79fddb9c20648a078
🏃 View run BEST_MODEL at: https://dagshub.com/IbrahimFaye/weather-agri.mlflow/#/experiments/3/runs/4e3936edd4bc40e79fddb9c20648a078
🧪 View experiment at: https://dagshub.com/IbrahimFaye/weather-agri.mlflow/#/experiments/3

📊 RAPPORT RÉCAPITULATIF DES MODÈLES AVEC MLFLOW

🎯 EXPÉRIENCES MLFLOW:
  - rain_forecast: Pré

# Results

In [41]:
import shutil
import os

# Archive the /content/result directory
output_filename = 'result_archive'
shutil.make_archive(output_filename, 'zip', '/content/result')

print(f"Archive created: {output_filename}.zip")
print("You can download it using the file browser on the left or with the following command:")
print(f"!google-colab-helper download {output_filename}.zip")

Archive created: result_archive.zip
You can download it using the file browser on the left or with the following command:
!google-colab-helper download result_archive.zip
